In [ ]:
import pandas as pd
import sys
from pathlib import Path

# Add parent directory to path to import athletics_performance
sys.path.insert(0, str(Path.cwd().parent))

from athletics_performance import ScoringTables, EVENT_CATALOG, Performance, Athlete
from datetime import date

In [ ]:
# Build parquet from PDF (run once)
# This parses the 846-page World Athletics Scoring Tables 2025 PDF
# and saves it as a local parquet file for fast lookups

print("Building parquet from PDF...")
print(f"PDF path: {ScoringTables.PDF_PATH}")
print(f"Target parquet: {ScoringTables.DATA_PATH}")

# Uncomment to build (takes ~30 seconds):
# ScoringTables.build_from_pdf()
# print("✓ Parquet built successfully!")

print("\nNote: PDF parsing extracts:")
print("  - 846 pages across 28 sections (14 Men + 14 Women)")
print("  - Track, field, combined events")
print("  - Points 1-1400 with corresponding performance thresholds")
print("  - Time format conversion (MM:SS.cc, H:MM:SS, decimal seconds)")
print("  - Result: ~70k rows (sex, event, points, performance)")

In [ ]:
# Load the scoring tables from parquet
st = ScoringTables.load()

print(f"✓ Loaded ScoringTables from {ScoringTables.DATA_PATH}")
print(f"\nDataframe shape: {st._df.shape}")
print(f"\nColumns: {list(st._df.columns)}")
print(f"\nDataframe preview:")
print(st._df.head(10))

In [ ]:
# Score some example performances
print("=" * 60)
print("SCORING EXAMPLES")
print("=" * 60)

# 100m men - Usain Bolt WR
pts_100m = st.score("M", "100m", 9.58)
print(f"\nUsain Bolt 100m WR (9.58s): {pts_100m} points")

# 100m women - Florence Griffith-Joyner WR
pts_100m_w = st.score("W", "100m", 10.49)
print(f"Florence Griffith-Joyner 100m WR (10.49s): {pts_100m_w} points")

# Long jump men - Mike Powell WR
pts_lj = st.score("M", "LJ", 8.95)
print(f"\nMike Powell LJ WR (8.95m): {pts_lj} points")

# Long jump women - Galina Chistyakova WR
pts_lj_w = st.score("W", "LJ", 7.52)
print(f"Galina Chistyakova LJ WR (7.52m): {pts_lj_w} points")

# Marathon men - Kelvin Kiptum WR
marathon_secs = (2 * 3600) + (1 * 60) + 9  # 2:01:09
pts_mar = st.score("M", "Mar", marathon_secs)
print(f"\nKelvin Kiptum Marathon WR (2:01:09): {pts_mar} points")

# Decathlon men - Kevin Mayer WR
pts_dec = st.score("M", "Dec", 9126)
print(f"Kevin Mayer Decathlon WR (9126 pts): {pts_dec} points")

print("\n" + "=" * 60)
print("PERFORMANCE FOR POINTS (Inverse Lookup)")
print("=" * 60)

# What time needed for 800 points in 100m (men and women)?
perf_m = st.performance_for_points("M", "100m", 800)
perf_w = st.performance_for_points("W", "100m", 800)
print(f"\nTime needed for 800 points:")
print(f"  Men 100m: {perf_m:.2f}s")
print(f"  Women 100m: {perf_w:.2f}s")

# What distance needed for 800 points in LJ?
perf_lj_m = st.performance_for_points("M", "LJ", 800)
perf_lj_w = st.performance_for_points("W", "LJ", 800)
print(f"\nDistance needed for 800 points:")
print(f"  Men LJ: {perf_lj_m:.2f}m")
print(f"  Women LJ: {perf_lj_w:.2f}m")

print("\n" + "=" * 60)
print("SCORE TABLE: 100M MEN (Points 1050-1100)")
print("=" * 60)

# Display a subset of the scoring table for 100m men
table_100m = st._df[
    (st._df['sex'] == 'M') & 
    (st._df['event'] == '100m') & 
    (st._df['points'] >= 1050) & 
    (st._df['points'] <= 1100)
].sort_values('points', ascending=False)

print(f"\n{len(table_100m)} entries in this range:\n")
print(table_100m[['points', 'performance']].to_string(index=False))

print("\n" + "=" * 60)
print("AVAILABLE EVENTS")
print("=" * 60)

# List all available events for men
events_m = st.available_events("M")
events_w = st.available_events("W")

print(f"\nMen's events ({len(events_m)} total):")
for i, event in enumerate(events_m[:20]):  # Show first 20
    print(f"  {event}", end="  ")
    if (i + 1) % 4 == 0:
        print()
if len(events_m) > 20:
    print(f"\n  ... and {len(events_m) - 20} more")

print(f"\n\nWomen's events ({len(events_w)} total):")
for i, event in enumerate(events_w[:20]):  # Show first 20
    print(f"  {event}", end="  ")
    if (i + 1) % 4 == 0:
        print()
if len(events_w) > 20:
    print(f"\n  ... and {len(events_w) - 20} more")

print("\n" + "=" * 60)
print("PERFORMANCE CLASS INTEGRATION")
print("=" * 60)

# Create a Performance instance and score it
athlete = Athlete(
    licence="2275784",
    last_name="Perrin",
    first_name="Guillaume",
    yob=1982,
    sex="M"
)

perf = Performance(
    perf_id="TEST_100M_001",
    date=date(2026, 4, 17),
    result_value=10.50,
    measurement="time",
    unit="s",
    athlete=athlete,
    event_id="100m",
    venue="Lyon"
)

pts = perf.score_points(st, "M")
print(f"\n{athlete.full_name} running 100m in {perf.result_value:.2f}s:")
print(f"  Performance ID: {perf.perf_id}")
print(f"  Date: {perf.date}")
print(f"  Venue: {perf.venue}")
print(f"  World Athletics Points: {pts}")
print(f"  Year of Season: {perf.yos}")